# Lab: Hồi quy Tuyến tính (Linear Regression)

## 1. Bài toán hồi quy là gì?

Khác với **phân loại** (classification) — đoán nhãn rời rạc, **hồi quy** (regression) đoán giá trị *liên tục*: giá nhà, nhiệt độ, doanh thu...

Ví dụ: cho diện tích nhà → đoán giá. Mỗi diện tích cho ra một số thực, không phải nhóm.

## 2. Mô hình tuyến tính

Mô hình đơn giản nhất: giả sử mối quan hệ giữa input $x = (x_1, x_2, \dots, x_n)$ và output $y$ là **tuyến tính**:
$$
\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = w^T x + b
$$

Mục tiêu: tìm $w$ và $b$ sao cho $\hat{y}$ gần $y$ thực tế nhất.

## 3. Hàm mất mát: Mean Squared Error

$$
L(w, b) = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2 = \frac{1}{N}\sum_{i=1}^{N}(w^T x_i + b - y_i)^2
$$

MSE phạt **bình phương** sai số → outlier ảnh hưởng rất mạnh. Nếu data có nhiều outlier, dùng **MAE** (Mean Absolute Error) hoặc **Huber loss** thay thế.

## 4. Hai cách tìm $(w, b)$

### 4.1. Normal Equation — công thức đóng (closed-form)

Đặt $X$ là ma trận $N \times (n+1)$ (thêm cột 1 cho bias), $y$ là vector $N \times 1$. Giải $\nabla L = 0$ cho ra:
$$
\hat{\theta} = (X^T X)^{-1} X^T y
$$

Đẹp, không cần lặp. Nhược: $O(n^3)$ để inverse → chậm khi nhiều feature.

### 4.2. Gradient Descent — lặp

Gradient của MSE:
$$\frac{\partial L}{\partial w_j} = \frac{2}{N}\sum_{i=1}^{N}(w^T x_i + b - y_i) \cdot x_{i,j}$$

Cập nhật:
$$w \leftarrow w - \alpha \nabla_w L, \quad b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

Chậm hơn về toán nhưng scale được lên dataset cực lớn. Đây là cách *neural network* học.

## 5. Đánh giá: $R^2$ và RMSE

**RMSE** (Root MSE): $\sqrt{L}$ — đơn vị = đơn vị của $y$ → dễ diễn giải.

**$R^2$ (coefficient of determination)**: tỷ lệ variance của $y$ được model giải thích.
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
$R^2 = 1$: hoàn hảo. $R^2 = 0$: bằng đoán mean. $R^2 < 0$: tệ hơn đoán mean.

---

## 0. Trước hết: hồi quy khác phân loại chỗ nào?

![Hồi quy so với phân loại](images/01_hoiquy_vs_phanloai.png)

*Bên trái — hồi quy: model trả về **một con số** trên trục liên tục. Bên phải — phân loại: model trả về **một nhãn** trong tập hữu hạn. Cùng là học có giám sát, nhưng hàm mất mát và cách đánh giá hoàn toàn khác nhau.*

| | Hồi quy | Phân loại |
|---|---|---|
| Output | Số thực $y \in \mathbb{R}$ | Nhãn $y \in \{1,\dots,C\}$ |
| Loss điển hình | MSE, MAE, Huber | Cross-Entropy, Hinge |
| Đánh giá | RMSE, MAE, $R^2$ | Accuracy, F1, AUC |
| Câu hỏi mẫu | "Căn nhà này giá bao nhiêu?" | "Email này có phải spam không?" |

**Bẫy hay gặp:** biến mục tiêu có vẻ là số nhưng thực chất là hạng (rating 1–5 sao, xếp loại A/B/C). Với dữ liệu như vậy, hồi quy tuyến tính "chạy được" nhưng giả định *khoảng cách giữa 4 và 5 sao bằng khoảng cách giữa 1 và 2 sao* — thường là sai. Khi đó nên dùng ordinal regression hoặc phân loại.

## 3b. Vì sao lại BÌNH PHƯƠNG sai số? — và khi nào đừng làm vậy

![Ý nghĩa hình học của MSE](images/02_mse_hinh_vuong_sai_so.png)

*Mỗi sai số $e_i = \hat{y}_i - y_i$ được biến thành một hình vuông cạnh $|e_i|$. MSE chính là **diện tích trung bình** của các hình vuông đó. Điểm cách đường 2 đơn vị đóng góp diện tích 4; điểm cách 4 đơn vị đóng góp 16 — gấp **4 lần**, dù chỉ xa gấp đôi.*

Ba lý do khiến MSE được chọn làm mặc định:

1. **Khả vi ở mọi nơi.** $|e|$ không có đạo hàm tại $e=0$, còn $e^2$ thì mượt → Gradient Descent chạy êm.
2. **Có nghiệm đóng.** Đạo hàm của tổng bình phương là tuyến tính theo tham số → giải $\nabla L = 0$ ra công thức Normal Equation. Với MAE thì không có may mắn đó.
3. **Có nền tảng thống kê.** Nếu nhiễu $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ độc lập, thì **tối thiểu MSE tương đương ước lượng hợp lý cực đại (MLE)**. Chứng minh ngắn:
$$
\log \mathcal{L}(\theta) = \sum_{i=1}^{N} \log \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(y_i - \theta^Tx_i)^2}{2\sigma^2}}
= \underbrace{-\frac{N}{2}\log(2\pi\sigma^2)}_{\text{hằng số}} - \frac{1}{2\sigma^2}\sum_{i=1}^{N}(y_i - \theta^Tx_i)^2
$$
Cực đại log-likelihood $\iff$ cực tiểu $\sum (y_i - \theta^Tx_i)^2$. Vậy **MSE không phải chọn tuỳ tiện** — nó là hệ quả của giả định nhiễu Gaussian.

### Mặt trái: MSE cực kỳ nhạy với outlier

![MSE, MAE và Huber trước outlier](images/09_outlier_mse_vs_huber.png)

*Chỉ 3 điểm outlier ở $y \approx 72$ đã kéo hệ số OLS lệch hẳn khỏi giá trị thật $w=3$, trong khi `HuberRegressor` gần như không nhúc nhích. Panel phải cho thấy vì sao: MSE (đỏ) bùng nổ theo bậc hai, còn Huber (xanh lá) chuyển sang tuyến tính khi $|e| > \delta$.*

$$
L_\delta(e) = \begin{cases}
\tfrac{1}{2}e^2 & \text{nếu } |e| \le \delta \quad \text{(mượt như MSE ở vùng gần)}\\[4pt]
\delta\left(|e| - \tfrac{1}{2}\delta\right) & \text{nếu } |e| > \delta \quad \text{(chỉ tuyến tính ở vùng xa)}
\end{cases}
$$

| Loss | Nhạy outlier | Khả vi | Ước lượng cái gì | Dùng khi |
|---|---|---|---|---|
| MSE | **Rất cao** | Mọi nơi | Trung bình có điều kiện $E[y|x]$ | Nhiễu Gaussian, ít outlier |
| MAE | Thấp | Trừ $e=0$ | **Trung vị** có điều kiện | Nhiều outlier, phân phối lệch |
| Huber | Thấp | Mọi nơi | Lai giữa hai cái trên | Mặc định an toàn khi nghi có outlier |
| Quantile (pinball) | Thấp | Trừ $e=0$ | Phân vị $\tau$ bất kỳ | Cần khoảng dự đoán, không chỉ điểm |

**Quy tắc thực hành:** luôn vẽ boxplot / histogram của $y$ trước khi chọn loss. Nếu thấy đuôi dài, hãy thử `HuberRegressor` hoặc `RANSACRegressor` song song với `LinearRegression`.

## 4.1b. Suy ra Normal Equation — và ý nghĩa hình học của nó

Viết loss dưới dạng ma trận (bỏ hệ số $1/N$ vì không đổi vị trí cực tiểu):
$$
L(\theta) = \|X\theta - y\|^2 = (X\theta - y)^T(X\theta - y) = \theta^T X^T X \theta - 2\theta^T X^T y + y^Ty
$$

Lấy gradient theo $\theta$ rồi cho bằng $0$:
$$
\nabla_\theta L = 2X^TX\theta - 2X^Ty = 0 \;\Longrightarrow\; \boxed{X^TX\hat{\theta} = X^Ty} \;\Longrightarrow\; \hat{\theta} = (X^TX)^{-1}X^Ty
$$

Phương trình $X^TX\theta = X^Ty$ được gọi là **normal equation** vì nó nói đúng một điều: $X^T(y - X\theta) = 0$, tức **vector phần dư vuông góc (normal) với mọi cột của $X$**.

![Ý nghĩa hình học của Normal Equation](images/05_normal_equation_hinh_chieu.png)

*Không gian cột của $X$ là mặt phẳng chứa mọi dự đoán khả dĩ $X\theta$. Vector $y$ nói chung **không nằm** trên mặt phẳng đó. Điểm gần $y$ nhất trên mặt phẳng chính là **hình chiếu vuông góc** của $y$ — và đó là $\hat{y}$. Phần dư $e = y - \hat{y}$ vuông góc với mặt phẳng: đó là toàn bộ nội dung của Normal Equation.*

Ma trận $H = X(X^TX)^{-1}X^T$ thực hiện phép chiếu này ($\hat{y} = Hy$) nên được gọi là **hat matrix**.

### Khi nào Normal Equation gặp rắc rối?

$(X^TX)^{-1}$ chỉ tồn tại khi $X$ **đủ hạng cột** (các cột độc lập tuyến tính). Ba tình huống hỏng:

| Tình huống | Triệu chứng | Cách chữa |
|---|---|---|
| Số feature > số mẫu ($n > N$) | $X^TX$ suy biến, vô số nghiệm | Ridge, hoặc giảm chiều (PCA) |
| Hai feature trùng nhau / cộng tuyến hoàn hảo (ví dụ one-hot đủ $C$ cột + cột bias) | Suy biến | Bỏ 1 cột (`drop_first=True`) |
| Cộng tuyến gần hoàn hảo (multicollinearity) | Nghịch đảo được nhưng hệ số khổng lồ, đổi dấu thất thường | Ridge, hoặc bỏ bớt feature theo VIF |

**Thực tế sklearn không gọi `np.linalg.inv`** — nó dùng phân rã SVD / `lstsq`, vừa ổn định số học hơn vừa trả về nghiệm chuẩn nhỏ nhất (pseudo-inverse Moore–Penrose) khi ma trận suy biến. Trong lab ta viết `inv` cho dễ hiểu, nhưng **code sản phẩm nên dùng `np.linalg.lstsq(X, y, rcond=None)`**.

### Normal Equation hay Gradient Descent?

| | Normal Equation | Gradient Descent |
|---|---|---|
| Số vòng lặp | 0 (một phát ra ngay) | Hàng trăm → hàng triệu |
| Chi phí | $O(n^3 + n^2N)$ — chết khi $n$ lớn | $O(nN)$ mỗi vòng |
| Cần chọn learning rate? | Không | **Có** (và rất nhạy) |
| Cần scale feature? | Không bắt buộc | **Bắt buộc** (xem mục dưới) |
| Ngưỡng thực dụng | $n \lesssim 10^4$ feature | $n$ bất kỳ, $N$ cực lớn |
| Dữ liệu chảy (streaming) | Không | Được (SGD online) |

Ngưỡng vàng để nhớ: **dưới ~10.000 feature thì dùng công thức đóng; trên đó dùng gradient**. Toàn bộ deep learning nằm ở cột bên phải.

## 4.2b. Nhìn thấy Gradient Descent đang làm gì

![Mặt mất mát và đường đi của Gradient Descent](images/03_mat_mat_mat_va_duong_di_gd.png)

*Bên trái: MSE vẽ theo hai tham số $(w, b)$ tạo thành một cái **bát lồi** (paraboloid). Bên phải: nhìn từ trên xuống. Mỗi bước GD đi ngược hướng gradient, tức **vuông góc với đường đồng mức tại điểm đó**. Chú ý hai màu của đường đi: **50 bước đầu (đỏ)** lao rất nhanh vào đáy khe, rồi **1450 bước sau (xanh)** bò chậm chạp dọc theo khe mới tới được nghiệm tối ưu.*

Vì sao lại chậm như vậy? Vì trong hình này feature $x$ chưa được chuẩn hoá (nằm trong khoảng 0 đến 10), khiến Hessian có **số điều kiện** $\kappa = \lambda_{\max}/\lambda_{\min} \approx 137$. Contour vì thế dẹt như một cái khe hẹp, và số bước GD cần tỷ lệ thuận với $\kappa$. Chỉ cần `StandardScaler` là $\kappa$ về gần 1 và bài toán này chỉ còn tốn vài chục bước. Chi tiết ở mục 4.2c.

Điểm mấu chốt về mặt lý thuyết: hàm mất mát MSE của hồi quy tuyến tính là **lồi** (convex) — ma trận Hessian $\nabla^2 L = \frac{2}{N}X^TX$ luôn nửa xác định dương. Hệ quả:

- Chỉ có **một** cực tiểu, không có local minimum để bị kẹt.
- Chọn learning rate đủ nhỏ thì GD **chắc chắn** hội tụ về nghiệm tối ưu toàn cục.

Đây là điều mà mạng neural **không** có: loss của MLP là non-convex, đầy local minima và saddle point. Nhưng vì hồi quy tuyến tính lồi, nó là bài toán hoàn hảo để học GD.

### Ba biến thể Gradient Descent

| Biến thể | Mỗi bước dùng | Ưu | Nhược |
|---|---|---|---|
| **Batch GD** | Toàn bộ $N$ mẫu | Gradient chính xác, đường đi mượt | Chậm khi $N$ lớn; không chạy được nếu data không vừa RAM |
| **Stochastic GD (SGD)** | **1** mẫu | Cực nhanh mỗi bước, thoát được vùng phẳng | Đường đi nhiễu, dao động quanh nghiệm |
| **Mini-batch GD** | $B$ mẫu (32–512) | Cân bằng tốt nhất; tận dụng vector hoá GPU | Thêm một hyperparameter $B$ |

Trong thực tế **mini-batch thắng tuyệt đối** — mọi framework deep learning đều mặc định dùng nó. Nhiễu của SGD/mini-batch không hẳn xấu: nó đóng vai trò như một dạng regularization nhẹ và giúp thoát saddle point.

Trong sklearn: `SGDRegressor(loss='squared_error')` là bản GD của `LinearRegression` — dùng khi dữ liệu quá lớn cho công thức đóng.

## 4.2c. Learning rate — tham số dễ làm hỏng việc nhất

![Ảnh hưởng của learning rate](images/04_anh_huong_learning_rate.png)

*Cùng một bài toán, cùng điểm khởi tạo, chỉ đổi $\alpha$. **Quá nhỏ (0.02)**: đi đúng hướng nhưng 40 bước vẫn chưa tới đích, MSE mới xuống 23.6. **Vừa (0.35)**: cắm thẳng vào tâm sau khoảng 15 bước, MSE = 1.36. **Quá lớn (1.02)**: mỗi bước vọt qua đáy sang sườn bên kia rồi văng ra xa dần, MSE **tăng** lên 11 460 và sẽ thành `inf`. Chú ý hàng dưới: đường loss của panel thứ ba đi LÊN, không phải đi xuống.*

Vì hình này dùng feature đã chuẩn hoá nên Hessian đúng bằng $2I$, tức $\lambda_{\max}=2$ và ngưỡng phân kỳ đúng bằng $2/\lambda_{\max} = 1.0$. Đó là lý do $\alpha = 0.35$ vẫn an toàn còn $\alpha = 1.02$ thì hỏng: không phải chuyện may rủi, mà là một con số tính được trước.*

Có một ngưỡng lý thuyết rõ ràng: GD trên hàm bậc hai hội tụ khi
$$
0 < \alpha < \frac{2}{\lambda_{\max}}
$$
với $\lambda_{\max}$ là trị riêng lớn nhất của Hessian $\frac{2}{N}X^TX$. Vượt ngưỡng này thì **chắc chắn** phân kỳ, không có ngoại lệ.

**Cách chẩn đoán khi train (thuộc lòng):**

| Triệu chứng đường loss | Chẩn đoán | Xử lý |
|---|---|---|
| Giảm rất chậm, gần như thẳng | $\alpha$ quá nhỏ | Nhân $\alpha$ lên 3–10 lần |
| Giảm nhanh rồi phẳng ngang | Đã hội tụ (hoặc kẹt) | Dừng lại, hoặc giảm $\alpha$ để tinh chỉnh |
| Dao động lên xuống quanh một mức | $\alpha$ hơi lớn | Giảm $\alpha$, hoặc dùng learning-rate decay |
| Tăng vọt, ra `nan` / `inf` | $\alpha$ quá lớn | Giảm $\alpha$ 10 lần và **scale feature** |

Chiến lược thực dụng: thử $\alpha \in \{10^{-1}, 10^{-2}, 10^{-3}, 10^{-4}\}$, chọn giá trị lớn nhất mà loss vẫn giảm ổn định, rồi thêm **learning rate decay** (giảm dần $\alpha$ theo thời gian) để tinh chỉnh ở cuối.

### Vì sao Gradient Descent BẮT BUỘC phải scale feature

Nhìn lại đường đi GD ở hình mục 4.2b: nó không cắm thẳng vào tâm mà lao vào khe rồi **bò rất chậm dọc theo khe**. Nguyên nhân: contour là hình **ellipse dẹt**, không phải hình tròn.

Khi các feature có thang đo lệch nhau (ví dụ `Age` ∈ [20, 70] và `Income` ∈ [20.000, 200.000]), Hessian $X^TX$ có **số điều kiện** (condition number) $\kappa = \lambda_{\max}/\lambda_{\min}$ rất lớn → contour dẹt như quả trứng → gradient hầu như luôn trỏ ngang qua khe hẹp thay vì dọc theo khe. Số bước cần để hội tụ tỷ lệ với $\kappa$.

`StandardScaler` kéo mọi feature về cùng thang $\mu=0, \sigma=1$ → contour tròn lại → GD đi gần như thẳng vào tâm. Đây là lý do **scale không phải "mẹo cho đẹp" mà là điều kiện cần để GD chạy nhanh**.

> **Ghi nhớ:** Normal Equation không cần scale (kết quả toán học y hệt). Gradient Descent và mọi mô hình có regularization thì **bắt buộc**.

## 5b. Bốn giả định của hồi quy tuyến tính — và cách kiểm tra bằng mắt

Hồi quy tuyến tính chỉ cho kết quả *đáng tin* khi bốn giả định sau tạm ổn (viết tắt tiếng Anh là **LINE**):

| Giả định | Nội dung | Vi phạm thì sao | Kiểm tra bằng |
|---|---|---|---|
| **L**inearity | Quan hệ giữa $x$ và $E[y]$ là tuyến tính | Model chệch có hệ thống | Residual plot có dạng cong |
| **I**ndependence | Các phần dư độc lập nhau | Sai số chuẩn bị đánh giá thấp → p-value sai | Durbin–Watson (dữ liệu chuỗi thời gian) |
| **N**ormality | Phần dư phân phối chuẩn | Khoảng tin cậy / kiểm định không còn đúng | Q–Q plot, histogram phần dư |
| **E**qual variance | Phương sai phần dư không đổi (homoscedasticity) | Ước lượng vẫn không chệch nhưng kém hiệu quả | Residual plot hình phễu |

Lưu ý quan trọng: giả định **Normality chỉ cần cho suy diễn thống kê** (khoảng tin cậy, p-value). Nếu bạn chỉ quan tâm **dự đoán**, vi phạm nó không nghiêm trọng. Còn Linearity và Equal variance thì ảnh hưởng trực tiếp đến chất lượng dự đoán.

![Ba dạng residual plot điển hình](images/06_chan_doan_phan_du.png)

*Cách đọc: vẽ phần dư theo giá trị dự đoán. **Trái** — mây điểm ngẫu nhiên quanh đường 0, không có hình thù: mô hình ổn. **Giữa** — hình chữ U rõ ràng: còn quan hệ phi tuyến chưa được model bắt, hãy thêm $x^2$ hoặc đổi sang model phi tuyến. **Phải** — hình cái phễu loe ra: phương sai tăng theo $\hat{y}$, hãy thử biến đổi $\log(y)$ hoặc dùng Weighted Least Squares.*

**Quy tắc số 1 khi làm hồi quy: vẽ residual plot trước khi khoe $R^2$.** Một $R^2 = 0.9$ đi kèm residual plot hình chữ U vẫn là một mô hình sai — nó chỉ đang sai *một cách có hệ thống* mà bạn chưa nhìn thấy.

## 5c. Đọc $R^2$ cho đúng

$R^2$ có một tính chất nguy hiểm: **thêm feature bất kỳ, kể cả feature ngẫu nhiên vô nghĩa, $R^2$ trên tập train không bao giờ giảm.** Vì vậy dùng $R^2$ để so sánh các model có số feature khác nhau là sai lầm.

**Adjusted $R^2$** vá lỗ hổng đó bằng cách phạt số feature:
$$
R^2_{\text{adj}} = 1 - (1 - R^2)\frac{N - 1}{N - n - 1}
$$
với $N$ = số mẫu, $n$ = số feature. Thêm feature vô dụng → $R^2$ tăng tí xíu nhưng mẫu số co lại → $R^2_{\text{adj}}$ **giảm**. Đó là tín hiệu "feature này không đáng giá".

| Chỉ số | Đơn vị | Đọc thế nào | Cẩn thận |
|---|---|---|---|
| **MSE** | (đơn vị $y$)² | Càng nhỏ càng tốt | Khó diễn giải vì bình phương |
| **RMSE** | Đơn vị $y$ | "Sai trung bình khoảng ± RMSE" | Vẫn nhạy outlier |
| **MAE** | Đơn vị $y$ | Sai số tuyệt đối trung bình | Không phạt nặng lỗi lớn |
| **$R^2$** | Không đơn vị | Tỷ lệ phương sai giải thích được | Luôn tăng khi thêm feature |
| **$R^2_{\text{adj}}$** | Không đơn vị | Như trên nhưng đã phạt số feature | Chỉ dùng cho tuyến tính |
| **MAPE** | % | Sai số phần trăm — dễ báo cáo cho sếp | **Nổ tung khi $y$ gần 0** |

Ba mốc cần thuộc: $R^2 = 1$ hoàn hảo; $R^2 = 0$ đúng bằng việc luôn đoán $\bar{y}$; $R^2 < 0$ **tệ hơn cả đoán trung bình** (rất hay gặp trên tập test khi model overfit).

# THỰC HÀNH 1: Hồi quy 1 biến với dữ liệu giả lập

Sinh dữ liệu $y = 3x + 5 + \varepsilon$, train cả 3 cách (Normal Equation, GD thủ công, sklearn) và so sánh.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(42)

# Sinh data y = 3x + 5 + nhiễu
N = 100
x = np.linspace(0, 10, N)
y = 3 * x + 5 + np.random.randn(N) * 1.5    # nhiễu N(0, 1.5)

plt.figure(figsize=(8, 4))
plt.scatter(x, y, alpha=0.6, label='Dữ liệu')
plt.plot(x, 3*x + 5, 'r-', label='True line: y = 3x + 5')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Dữ liệu giả lập')
plt.show()

### Cách 1: Normal Equation

In [ ]:
# Thêm cột 1 cho bias: X có shape (N, 2), cột đầu là 1, cột sau là x.
X = np.column_stack([np.ones(N), x])
theta_ne = np.linalg.inv(X.T @ X) @ X.T @ y
b_ne, w_ne = theta_ne
print(f'Normal Equation: w = {w_ne:.4f},  b = {b_ne:.4f}')
print(f'                 (so với true: w=3, b=5)')

### Cách 2: Gradient Descent thủ công

In [ ]:
w, b = 0.0, 0.0
lr = 0.01
loss_history = []

for step in range(1000):
    y_hat = w * x + b
    err = y_hat - y
    loss = (err ** 2).mean()
    loss_history.append(loss)

    # gradient
    dw = 2 * (err * x).mean()
    db = 2 * err.mean()
    w -= lr * dw
    b -= lr * db

print(f'Gradient Descent: w = {w:.4f},  b = {b:.4f}')
print(f'Final loss: {loss:.4f}')

plt.figure(figsize=(8, 3.5))
plt.plot(loss_history); plt.xlabel('Step'); plt.ylabel('MSE'); plt.yscale('log')
plt.title('Loss giảm theo số bước GD'); plt.grid(alpha=0.3); plt.show()

### Cách 3: sklearn

In [ ]:
model = LinearRegression()
model.fit(x.reshape(-1, 1), y)
print(f'sklearn:         w = {model.coef_[0]:.4f},  b = {model.intercept_:.4f}')

# Vẽ ba đường
plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.5, label='Data')
plt.plot(x, 3*x + 5, 'r-', linewidth=2, label='True')
plt.plot(x, w_ne*x + b_ne, 'g--', linewidth=2, label='Normal Equation')
plt.plot(x, w*x + b, 'b:', linewidth=2, label='Gradient Descent')
plt.plot(x, model.coef_[0]*x + model.intercept_, 'm-.', linewidth=2, label='sklearn')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print('Cả 3 cách cho ra gần như cùng kết quả (chỉ lệch nhỏ ở GD nếu chưa đủ vòng).')

# THỰC HÀNH 2: Hồi quy đa biến — California Housing

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X, y = housing.data, housing.target
feat_names = housing.feature_names
print(f'Shape: X={X.shape}, y={y.shape}')
print(f'Features: {feat_names}')
print(f'Target = giá nhà (đơn vị: trăm nghìn USD)')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scaling: fit chỉ trên train để tránh data leakage.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f'Test RMSE: {rmse:.3f}  (đơn vị $100K)')
print(f'Test MAE : {mae:.3f}')
print(f'Test R^2 : {r2:.3f}')

# Hệ số
coef = pd.Series(model.coef_, index=feat_names).sort_values()
coef.plot.barh(figsize=(7, 4)); plt.xlabel('Hệ số (đã chuẩn hoá)')
plt.title('Tác động của từng feature lên giá nhà'); plt.tight_layout(); plt.show()

In [ ]:
# Vẽ predicted vs actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Hoàn hảo')
plt.xlabel('Giá thật'); plt.ylabel('Giá dự đoán')
plt.title(f'Predicted vs Actual (R² = {r2:.3f})')
plt.legend(); plt.grid(alpha=0.3)
plt.axis('equal'); plt.show()

In [ ]:
# Kiểm tra giả định của model California Housing vừa train ở trên.
# Đây là bước mà sinh viên hay bỏ qua — nhưng nó nói nhiều hơn con số R^2.
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

# (1) Residual vs Fitted — kiểm tra linearity + homoscedasticity
axes[0].scatter(y_pred, residuals, s=8, alpha=0.3)
axes[0].axhline(0, color='red', lw=1.5)
axes[0].set_xlabel('Giá trị dự đoán'); axes[0].set_ylabel('Phần dư')
axes[0].set_title('Residual vs Fitted')

# (2) Histogram phần dư — kiểm tra normality
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Phần dư'); axes[1].set_ylabel('Tần suất')
axes[1].set_title('Phân phối phần dư')

# (3) Q-Q plot — normality chặt chẽ hơn histogram
from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q plot')

plt.tight_layout(); plt.show()

# Adjusted R^2
n_samples, n_features = X_test_s.shape
r2_adj = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
print(f'R^2          = {r2:.4f}')
print(f'Adjusted R^2 = {r2_adj:.4f}   ({n_features} feature, {n_samples} mẫu test)')
print()
print('Quan sát: phần dư có xu hướng cụp xuống ở vùng giá cao — vì target California')
print('Housing bị CẮT NGỌN ở mức 5.0 ($500K). Model không thể đoán quá 5 nên phần dư lệch.')
print('Đây là ví dụ điển hình: residual plot phát hiện vấn đề mà R^2 giấu đi.')

## 6. Regularization — Ridge và Lasso

Hồi quy thường gặp vấn đề khi feature có tương quan cao (multicollinearity) → hệ số bùng nổ. **Regularization** thêm penalty vào loss để khống chế:

**Ridge (L2)**: $L = \text{MSE} + \alpha \sum w_j^2$. Co hệ số về 0 nhưng không chính xác là 0.

**Lasso (L1)**: $L = \text{MSE} + \alpha \sum |w_j|$. Có thể đẩy hệ số về *đúng 0* → tự động chọn feature.

### Hình học đằng sau Ridge và Lasso: vì sao chỉ Lasso mới đẩy được hệ số về 0?

![Ridge so với Lasso](images/08_ridge_vs_lasso.png)

*Cách đọc hai panel trái: các ellipse đỏ là đường đồng mức của MSE (tâm = nghiệm OLS không bị phạt). Vùng xanh là **miền khả thi** do regularization áp đặt. Nghiệm bị phạt là điểm mà ellipse chạm vùng xanh lần đầu.*

- **Ridge (L2)** có miền khả thi là **hình tròn** — trơn tru, không góc cạnh. Ellipse hầu như luôn chạm vào một điểm có cả hai toạ độ khác 0 → hệ số bị **co nhỏ** nhưng không bao giờ đúng bằng 0.
- **Lasso (L1)** có miền khả thi là **hình thoi** — có **góc nhọn nằm ngay trên trục toạ độ**. Ellipse rất dễ chạm đúng vào góc → nghiệm rơi vào trục → hệ số đó bằng **chính xác 0**.

Panel phải minh hoạ hệ quả: khi tăng $\alpha$, các hệ số của Lasso lần lượt "rụng" về 0 — đó chính là **chọn feature tự động** (embedded feature selection).

| | Ridge (L2) | Lasso (L1) | ElasticNet |
|---|---|---|---|
| Penalty | $\alpha\sum w_j^2$ | $\alpha\sum \lvert w_j \rvert$ | Kết hợp cả hai |
| Hệ số về đúng 0? | Không | **Có** | Có |
| Feature tương quan cao | Chia đều trọng số cho cả nhóm | Chọn **bừa một cái**, bỏ phần còn lại | Giữ cả nhóm (grouping effect) |
| $n > N$ (nhiều feature hơn mẫu) | Vẫn chạy | Chọn tối đa $N$ feature | Không giới hạn đó |
| Nghiệm đóng | **Có**: $(X^TX + \alpha I)^{-1}X^Ty$ | Không (phải dùng coordinate descent) | Không |
| Khi nào dùng | Mọi feature đều có ích chút ít | Tin rằng phần lớn feature là vô dụng | Nhiều feature và chúng tương quan nhau |

Một cách nhìn khác (Bayes): Ridge = đặt **prior Gaussian** lên hệ số, Lasso = đặt **prior Laplace** (nhọn tại 0 nên "kéo" hệ số về 0). Nghiệm chính là ước lượng MAP.

Ridge còn có một tác dụng phụ rất quý: $(X^TX + \alpha I)$ **luôn khả nghịch** kể cả khi $X^TX$ suy biến — nên Ridge cứu được trường hợp multicollinearity hoàn hảo mà OLS chịu thua.

> ⚠️ **Bắt buộc scale trước khi regularize.** Penalty cộng trực tiếp độ lớn hệ số. Feature đo bằng mét và feature đo bằng milimét sẽ bị phạt hoàn toàn khác nhau dù mang cùng thông tin. Luôn dùng `make_pipeline(StandardScaler(), Ridge(alpha=...))`.

In [ ]:
for name, mdl in [('Linear', LinearRegression()),
                  ('Ridge α=1', Ridge(alpha=1.0)),
                  ('Ridge α=10', Ridge(alpha=10.0)),
                  ('Lasso α=0.1', Lasso(alpha=0.1)),
                  ('Lasso α=1', Lasso(alpha=1.0))]:
    mdl.fit(X_train_s, y_train)
    pred = mdl.predict(X_test_s)
    r2 = r2_score(y_test, pred)
    nz = np.sum(np.abs(mdl.coef_) > 1e-6)
    print(f'{name:15s}  R² = {r2:.4f}  | Số coef khác 0: {nz}/{len(mdl.coef_)}')

## 7. Polynomial regression — học quan hệ phi tuyến

Linear Regression không *bắt buộc* dữ liệu phải tuyến tính theo $x$. Ta có thể tạo feature mới $x^2, x^3$ rồi chạy linear regression — đây là **polynomial regression**.

### Bias–Variance: khung tư duy đằng sau mọi lựa chọn độ phức tạp

![Bias-variance trade-off](images/07_bias_variance_tradeoff.png)

*Ba panel trái: cùng 18 điểm train, chỉ đổi bậc đa thức. **Degree 1** quá cứng, bỏ sót hẳn hình dạng sin, **bias cao**. **Degree 6** bám gần như trùng hàm thật. **Degree 17** uốn éo bám theo từng hạt nhiễu và vọt lên ở hai mép, **variance cao**. Panel phải: train RMSE đi xuống rồi nằm phẳng, còn test RMSE tạo hình chữ U, đáy chữ U ở bậc 6 chính là điểm nên dừng.*

> 🔧 **Một mẹo kỹ thuật ẩn trong hình này.** Pipeline phải là `PolynomialFeatures` → `StandardScaler` → `LinearRegression`. Nếu bỏ `StandardScaler`, ma trận thiết kế (dạng Vandermonde) bị suy biến số học ở bậc cao, và bạn sẽ thấy **train RMSE lại tăng** ở bậc lớn. Đó là hiện tượng *số học*, không phải hiện tượng *thống kê*, nhưng rất dễ khiến người đọc kết luận nhầm. Đây cũng là lý do sinh viên hay vẽ ra đường bias-variance kỳ quặc rồi không hiểu vì sao.

Sai số kỳ vọng của model trên một điểm mới tách được thành ba phần:
$$
\underbrace{E\big[(y - \hat{f}(x))^2\big]}_{\text{sai số kỳ vọng}} = \underbrace{\big(E[\hat{f}(x)] - f(x)\big)^2}_{\text{Bias}^2} + \underbrace{E\big[(\hat{f}(x) - E[\hat{f}(x)])^2\big]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{nhiễu không thể khử}}
$$

- **Bias**: model quá đơn giản, sai một cách có hệ thống dù cho bao nhiêu dữ liệu cũng vậy.
- **Variance**: model quá linh hoạt, đổi tập train một chút là kết quả đổi hẳn.
- **$\sigma^2$**: nhiễu bản chất của dữ liệu — **không mô hình nào vượt qua được**. Đây là trần hiệu năng lý thuyết.

| Triệu chứng | Chẩn đoán | Cách chữa |
|---|---|---|
| Train error CAO, test error CAO (xấp xỉ nhau) | **Underfit** (bias cao) | Thêm feature, tăng bậc/độ sâu, giảm regularization, train lâu hơn |
| Train error THẤP, test error CAO (khoảng cách lớn) | **Overfit** (variance cao) | Thêm dữ liệu, tăng regularization, giảm độ phức tạp, early stopping |
| Cả hai thấp và sát nhau | Ổn 👍 | Dừng lại và đi ăn mừng |
| Train error cao hơn test error | Nghi có bug hoặc rò rỉ dữ liệu | Kiểm tra lại quy trình split/scale |

**Ba cách kéo giảm variance mà không tăng bias** (ghi nhớ suốt môn học): (1) thêm dữ liệu, (2) regularization, (3) ensemble — đúng ba thứ ta sẽ dùng lại ở bài Random Forest.

> **Bẫy chết người:** tuyệt đối không dùng tập test để chọn bậc đa thức rồi lại báo cáo kết quả trên chính tập test đó. Làm vậy tập test đã trở thành tập validation, và con số bạn báo cáo là lạc quan giả tạo. Hãy dùng cross-validation trên tập train để chọn, rồi mới chạm vào test **đúng một lần**.

In [ ]:
# Sinh data phi tuyến: y = sin(x) + nhiễu
x_nl = np.sort(np.random.uniform(0, 2*np.pi, 100))
y_nl = np.sin(x_nl) + np.random.randn(100) * 0.15

plt.figure(figsize=(10, 4))
plt.scatter(x_nl, y_nl, alpha=0.5, label='Data')
for deg in [1, 3, 9]:
    mdl = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    mdl.fit(x_nl.reshape(-1, 1), y_nl)
    xx = np.linspace(0, 2*np.pi, 200).reshape(-1, 1)
    plt.plot(xx, mdl.predict(xx), label=f'degree={deg}', linewidth=2)
plt.legend(); plt.grid(alpha=0.3); plt.title('Polynomial regression bậc khác nhau')
plt.show()
print('degree=1: underfit (đường thẳng)')
print('degree=3: vừa (gần với sin trong khoảng dữ liệu)')
print('degree=9: bắt đầu lượn theo nhiễu — overfit')

## Tổng kết

1. Linear Regression: $\hat{y} = w^T x + b$, tối thiểu MSE.
2. Hai cách giải: Normal Equation (đóng) và Gradient Descent (lặp).
3. Đánh giá: RMSE, MAE, R².
4. **Ridge / Lasso** giúp chống overfitting và (Lasso) chọn feature.
5. **Polynomial regression**: thêm $x^2, x^3, ...$ để học quan hệ phi tuyến — nhưng coi chừng overfit.
6. **Phải scale feature** trước khi dùng regularization (vì penalty so với độ lớn coef).

# BÀI TẬP VỀ NHÀ

## Bài 1: Cài đặt từ scratch
Viết class `MyLinearRegression` với 2 method:
- `fit(X, y)` dùng Normal Equation.
- `predict(X)` dùng coefficient đã fit.

Test trên California Housing. So sánh hệ số và R² với `sklearn.LinearRegression`. Phải gần như giống hệt.

*Gợi ý:* nhớ thêm cột 1 cho bias trước khi inverse.

## Bài 2: Diện tích → giá nhà
Sinh dữ liệu giả lập:
- 200 căn nhà.
- Diện tích: từ 30m² đến 200m².
- Giá = 50 + 30·area + nhiễu (đơn vị triệu).

Train Linear Regression. Vẽ scatter + đường hồi quy. Báo cáo RMSE.

## Bài 3: Outlier ảnh hưởng MSE thế nào?
Lấy data `y = 3x + 5 + nhiễu` ở Lab 1. Thêm 5 outlier mạnh (y = 100). Train Linear Regression.

Sau đó dùng `HuberRegressor` (`from sklearn.linear_model import HuberRegressor`) — robust với outlier. So sánh hệ số học được. Cái nào sát true (3, 5) hơn?

## Bài 4: Ridge vs Lasso khi có feature thừa
Sinh dữ liệu: 100 mẫu, 20 feature, nhưng *chỉ 5 feature thật sự liên quan* đến y (15 còn lại là nhiễu).

1. Train `LinearRegression`, `Ridge(α=1)`, `Lasso(α=0.1)`.
2. Đếm số coef khác 0 sau train.
3. Lasso có "phát hiện" được 5 feature thật không?

*Gợi ý:* dùng `np.abs(model.coef_) > 1e-6` để đếm.

## Bài 5: Polynomial degree → overfit?
Trên data sin ở Lab cuối cùng:
1. Train polynomial regression với `degree ∈ {1, 2, ..., 15}`.
2. Với mỗi degree, chia 80/20 train/test, đo RMSE trên cả train và test.
3. Vẽ 2 đường RMSE theo degree.
4. Quan sát: train RMSE giảm liên tục, test RMSE giảm rồi tăng — đó là *bias-variance trade-off*. Tại degree nào test RMSE thấp nhất?